- LangChain 是一套面向大模型的开发框架
- LangChain 是 AGI 时代软件工程的一个探索和原型
- LangChain 并不完美，还在不断迭代中：我写这个课件的时候是 V0.0.200，第一期上课时是 V0.0.241，现在是V0.0.292
- 学习 LangChain 更重要的是借鉴其思想，具体的接口和模块可能很快就会改变


## LangChain 的核心组件

1. 模型 I/O 封装
   - LLMs：大语言模型
   - Chat Models：一般基于 LLMs，但按对话结构重新封装
   - PromptTemple：提示词模板
   - OutputParser：解析输出
2. 数据连接封装
   - Document Loaders：各种格式文件的加载器
   - Document transformers：对文档的常用操作，如：split, filter, translate, extract metadata, etc
   - Text Embedding Models：文本向量化表示，用于检索等操作（啥意思？别急，后面详细讲）
   - Verctor stores: （面向检索的）向量的存储
   - Retrievers: 向量的检索
3. 记忆封装
   - Memory：这里不是物理内存，从文本的角度，可以理解为“上文”、“历史记录”或者说“记忆力”的管理
4. 架构封装
   - Chain：实现一个功能或者一系列顺序功能组合
   - Agent：根据用户输入，自动规划执行步骤，自动选择每步需要的工具，最终完成用户指定的功能
     - Tools：调用外部功能的函数，例如：调 google 搜索、文件 I/O、Linux Shell 等等
     - Toolkits：操作某软件的一组工具集，例如：操作 DB、操作 Gmail 等等
5. Callbacks


## 一、模型 I/O 封装

### 1.1 模型 API：LLM vs. ChatModel


### 1.1.1 生成模型封装

In [1]:
import warnings
warnings.filterwarnings("ignore")
from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ["OPENAI_API_KEY"]


llm = OpenAI(openai_api_key=api_key)  # 默认是text-davinci-003模型
llm.predict("八百标兵奔北坡")

'\n\n八百标兵奔北坡，一片雄关殷勤开。帅气长戟势不可挡，豪迈冲锋英雄来。谦逊谨慎多谋略，准备充分定大局。紧箍咒语锁咆哮，铁骑飞扬护佑路。英勇无畏不停歇，坚强不屈守初心。勇往直前不停歇，追求真理安静远。'

### 1.1.2 对话模型封装

In [2]:
chat_model = ChatOpenAI()  # 默认是gpt-3.5-turbo
chat_model.predict("八百标兵奔北坡")

'八百标兵奔北坡，炮兵并排北边跑。\n炮兵急速拉火炮，标兵奔跑如飞快。\n\n北坡险峻标兵多，勇敢前进不畏难。\n冲锋炮声震山谷，战士们壮胆无畏惧。\n\n敌军阵地已被攻破，胜利在望士气高。\n八百标兵奔北坡，英勇壮举令人敬佩。'

### 1.1.3 多轮对话Session封装

In [3]:
from langchain.schema import (
    AIMessage, #等价于OpenAI接口中的assistant role
    HumanMessage, #等价于OpenAI接口中的user role
    SystemMessage #等价于OpenAI接口中的system role
)

messages = [
    SystemMessage(content="你的角色是github网站，你对自己的里面的项目资源十分清楚，你知道每个项目是做什么的。"), 
    HumanMessage(content="github上开源的项目open-interpreter是做什么的？") 
]
chat_model(messages) 

AIMessage(content='open-interpreter是一个开源的解释器项目，它的目标是提供一个可扩展的、跨平台的解释器框架，使开发者能够更轻松地构建和运行自己的脚本语言或领域特定语言（DSL）。这个项目提供了一些核心的解释器功能，如词法分析、语法分析、解释执行等，同时也提供了一些示例语言的实现，可以作为参考或直接使用。通过使用open-interpreter，开发者可以快速构建自己的解释器，并在不同的平台上运行。', additional_kwargs={}, example=False)

不同模型统一接口调用

In [4]:
from langchain.chat_models import ErnieBotChat
from langchain.schema import HumanMessage

chat_model = ErnieBotChat()

messages = [
    HumanMessage(content="你是谁") 
]

chat_model(messages)

AIMessage(content='我是百度公司开发的文心一言，英文名是ERNIE Bot，可以协助您完成范围广泛的任务并提供有关各种主题的信息，比如回答问题，提供定义和解释及建议。如果您有任何问题，请随时向我提问。', additional_kwargs={}, example=False)

### 1.2 模型的输入与输出

<img src="model_io.jpg" style="margin-left: 0px" width=500px>

### 1.2.1 Prompt模板封装

PromptTemplate

In [12]:
from langchain.prompts import PromptTemplate

template = PromptTemplate.from_template("{subject} 这个浏览器插件时做什么的？")
print(template.input_variables)
print(template.format(subject='篡改猴'))


['subject']
篡改猴 这个浏览器插件时做什么的？


ChatPromptTemplate

In [3]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.chat import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import ChatOpenAI

import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ["OPENAI_API_KEY"]

template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("你是{information}插件市场的客服助手。你对{information}插件市场上的每一个应用都熟记于心。"),
        HumanMessagePromptTemplate.from_template("{question} 这个插件是做什么的？"),
    ]
)
# llm=ChatOpenAI(temperature=0,openai_api_key=api_key, model_name="gpt-3.5-turbo")
llm=ChatOpenAI(openai_api_key=api_key)

llm(template.format_messages(
        information="Microfst Edge",
        question="篡改猴"
    ))

AIMessage(content='"篡改猴"是一款用于浏览器的插件，它可以修改网页的样式和行为。通过篡改猴，用户可以自定义网页的外观和功能，比如更改颜色、字体、图标，隐藏或显示元素，添加自定义脚本等。这个插件非常灵活，可以根据用户的个人喜好和需求来定制网页，提供更好的用户体验。', additional_kwargs={}, example=False)

<div class="alert alert-warning">
<b>思考：</b>把Prompt模板看作带有参数的函数，下面的内容可能更好理解
</div>


FewShotPromptTemplate

In [14]:
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.prompts import PromptTemplate

#例子(few-shot)
examples = [
    {
        "input": "北京天气怎么样",
        "output" : "北京市"
    },
    {
        "input": "南京下雨吗",
        "output" : "南京市"
    },
    {
        "input": "江城热吗",
        "output" : "武汉市"
    }
]

#例子拼装的格式
example_prompt = PromptTemplate(input_variables=["input", "output"], template="Input: {input}\nOutput: {output}")

#Prompt模板
prompt = FewShotPromptTemplate(
    examples=examples, 
    example_prompt=example_prompt, 
    suffix="Input: {input}\nOutput:", 
    input_variables=["input"]
)

prompt = prompt.format(input="羊城多少度")

print("===Prompt===")
print(prompt)

llm = OpenAI()
response = llm(prompt)

print("===Response===")
print(response)

===Prompt===
Input: 北京天气怎么样
Output: 北京市

Input: 南京下雨吗
Output: 南京市

Input: 江城热吗
Output: 武汉市

Input: 羊城多少度
Output:
===Response===
 广州市


### 1.3 OutputParser

### 1.3.1 Pydantic (JSON) Parser

自动根据Pydantic类的定义，生成输出的格式说明

In [2]:
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI

from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, validator
from typing import List, Dict
import json

# 避免print时中文变成unicode码
def chinese_friendly(string):
    lines = string.split('\n')
    for i, line in enumerate(lines):
        if line.startswith('{') and line.endswith('}'):
            try:
                lines[i] = json.dumps(json.loads(line), ensure_ascii=False)
            except:
                pass
    return '\n'.join(lines)


model_name = 'gpt-3.5-turbo-16k-0613'
temperature = 0
model = OpenAI(model_name=model_name, temperature=temperature)

# 定义你的输出格式
class Command(BaseModel):
    command: str = Field(description="linux shell命令名")
    arguments: Dict[str, str] = Field(description="命令的参数 (name:value)")

    # 你可以添加自定义的校验机制
    @validator('command')
    def no_space(cls, info):
        if " " in info or "\t" in info or "\n" in info:
            raise ValueError("命令名中不能包含空格或回车!")
        return info

# 根据Pydantic对象的定义，构造一个OutputParser
parser = PydanticOutputParser(pydantic_object=Command)

prompt = PromptTemplate(
    template="将用户的指令转换成linux命令.\n{format_instructions}\n{query}",
    input_variables=["query"],
    # 直接从OutputParser中获取输出描述，并对模板的变量预先赋值
    partial_variables={"format_instructions": parser.get_format_instructions()} 
)

print("====Format Instruction=====")
print(chinese_friendly(parser.get_format_instructions()))


query = "将系统日期设为2023-04-01"
model_input = prompt.format_prompt(query=query)

print("====Prompt=====")
print(chinese_friendly(model_input.to_string()))

output = model(model_input.to_string())
print("====Output=====")
print(output)
print("====Parsed=====")
cmd = parser.parse(output)
print(cmd)

====Format Instruction=====
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"command": {"title": "Command", "description": "linux shell命令名", "type": "string"}, "arguments": {"title": "Arguments", "description": "命令的参数 (name:value)", "type": "object", "additionalProperties": {"type": "string"}}}, "required": ["command", "arguments"]}
```
====Prompt=====
将用户的指令转换成linux命令.
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "ar

### 1.3.2 Auto-Fixing Parser

利用LLM自动根据解析异常修复并重新解析

In [21]:
from langchain.output_parsers import OutputFixingParser

new_parser = OutputFixingParser.from_llm(parser=parser, llm=ChatOpenAI())

#我们把之前output的格式改错
output = output.replace("\"","'")
print("===格式错误的Output===")
print(output)
try:
    cmd = parser.parse(output)
except Exception as e:
    print("===出现异常===")
    print(e)
    
#用OutputFixingParser自动修复并解析
cmd = new_parser.parse(output)
print("===重新解析结果===")
print(cmd)

===格式错误的Output===
{
  'command': 'date',
  'arguments': {
    'date': '2023-04-01'
  }
}
===出现异常===
Failed to parse Command from completion {
  'command': 'date',
  'arguments': {
    'date': '2023-04-01'
  }
}. Got: Expecting property name enclosed in double quotes: line 2 column 3 (char 4)
===重新解析结果===
command='date' arguments={'date': '2023-04-01'}


<div class="alert alert-warning">
<b>思考：</b>猜一下OutputFixingParser是怎么做到的
</div>

## 二、数据连接封装

<img src="data_connection.jpg" style="margin-left: 0px" width=500px>

### 2.1 文档加载器：Document Loaders


In [ ]:
!pip install pypdf


In [11]:
from langchain.document_loaders import PyPDFLoader

loader = PyPDFLoader("llama2.pdf")
pages = loader.load_and_split()

print(pages[0].page_content)

Llama 2 : Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗Louis Martin†Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic
Sergey Edunov 

### 2.2 文档处理器

### 2.2.1 TextSplitter


In [2]:
import re, wordninja

#预处理字符全都连在一起的行
def preprocess(text):
    def split(line):
        tokens = re.findall(r'\w+|[.,!?;%$-+=@#*/]', line)
        return [
            ' '.join(wordninja.split(token)) if token.isalnum() else token
            for token in tokens
        ]

    lines = text.split('\n')
    for i,line in enumerate(lines):
        if len(max(line.split(' '), key = len)) >= 20: 
            lines[i] = ' '.join(split(line))
    return ' '.join(lines)

In [13]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,  # 思考：为什么要做overlap
    length_function=len,
    add_start_index=True,
)

paragraphs = text_splitter.create_documents([preprocess(pages[3].page_content)])
for para in paragraphs:
    print(para.page_content)
    print('-------')

Figure 3: Safety human evaluation results for Llama 2-Chat compared to other open-source and closed- source models. Human raters judged model generations for safety violations across ~2,000
-------
generations for safety violations across ~2,000 adversarial prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is important to caveat these safety
-------
4.4. It is important to caveat these safety results with the inherent bias of LL M evaluations due to limitations of the prompt set , subjectivity of the review guidelines , and subjectivity of
-------
of the review guidelines , and subjectivity of individual rate rs . Additionally , these safety evaluations are performed using content standards that are likely to be biased towards the Llama
-------
that are likely to be biased towards the Llama 2-Chatmodels. We are releasing the following models to the general public for research and commercial use‡: 1 . Llama 2 , an updated version of L

### 2.2.2 Doctran


In [52]:
%pip install doctran #支持翻译功能

In [15]:
from langchain.document_transformers import DoctranTextTranslator

translator = DoctranTextTranslator(
    openai_api_model="gpt-3.5-turbo", language="Chinese"
)

translated_document = await translator.atransform_documents([pages[3]])

print(translated_document[0].page_content)

图3：Llama 2-Chat与其他开源和闭源模型的安全人工评估结果。人工评估员对大约2000个对抗性提示进行了安全违规的模型生成评判，包括单轮和多轮提示。更多细节请参见第4.4节。需要注意的是，由于提示集的限制、审查指南的主观性和个体评估员的主观性，这些安全评估结果可能存在固有的LLM评估偏差。此外，这些安全评估是使用可能对Llama 2-Chat模型有偏见的内容标准进行的。我们向公众发布以下模型供研究和商业用途‡：1. Llama 2，Llama 1的更新版本，使用新的公开可用数据进行训练。我们还将预训练语料库的大小增加了40％，将模型的上下文长度加倍，并采用了分组查询注意力（Ainslie等，2023）。我们发布了7B、13B和70B参数的Llama 2变体。我们还训练了34B变体，在本文中进行了报告，但不发布§。2. Llama 2-Chat，Llama 2的经过微调的版本，针对对话使用案例进行了优化。我们发布了7B、13B和70B参数的Llama 2-Chat变体。我们相信，安全地公开LLM将对社会产生净利益。像所有LLM一样，Llama 2是一项新技术，使用时存在潜在风险（Bender等，2021b; Weidinger等，2021; Solaiman等，2023）。迄今为止进行的测试是用英语进行的，无法涵盖所有场景。因此，在部署Llama 2-Chat的任何应用程序之前，开发人员应根据其特定的模型应用进行安全测试和调优。我们提供了一个负责任的使用指南¶和代码示例‖，以促进Llama 2和Llama 2-Chat的安全部署。有关我们负责任发布策略的更多细节，请参见第5.3节。本文的其余部分描述了我们的预训练方法（第2节），微调方法（第3节），模型安全方法（第4节），关键观察和见解（第5节），相关工作（第6节）和结论（第7节）。‡https://ai.meta.com/resources/models-and-libraries/llama/§由于缺乏足够的时间进行充分的红队测试，我们推迟了34B模型的发布。¶https://ai.meta.com/llama‖https://github.com/facebookresearch/llama


### 2.3 检索与问答

In [16]:
from langchain.retrievers import TFIDFRetriever  # 最传统的关键字加权检索
from langchain.text_splitter import RecursiveCharacterTextSplitter
import wordninja, re

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=60,  
    length_function=len,
    add_start_index=True,
)

# 取一个有信息量的章节（Introduction: 第2-3页）
paragraphs = text_splitter.create_documents(
    [preprocess(d.page_content) for d in pages[2:4]]
)

user_query = "Does llama 2 have a dialogue version?"

retriever = TFIDFRetriever.from_documents(paragraphs)
docs = retriever.get_relevant_documents(user_query)

print(docs[0].page_content)

34 B variants , which were port on in this paper but are not releasing.§ 2.Llama 2-Chat , a fine-tuned version of Llama 2 that is optimized for dialogue use cases. We release variants of this model


这里，我们暂时先手写一个问答过程

In [17]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.chat import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import ChatOpenAI

template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("你是问答机器人，你根据以下信息回答用户问题。\n已知信息：\n{information}\n\nBe brief, and do not make up information."),
        HumanMessagePromptTemplate.from_template("{query}"),
    ]
)

llm = ChatOpenAI(temperature=0)
response = llm(
            template.format_messages(
                information=docs[0].page_content,
                query=user_query
            )
        )
print(response.content)

Yes, Llama 2 has a dialogue version called Llama 2-Chat. It is a fine-tuned version of Llama 2 that is optimized for dialogue use cases.


In [18]:
#我们换个问法
user_query = "Does llama 2 have a conversational variant?"

retriever = TFIDFRetriever.from_documents(paragraphs)
docs = retriever.get_relevant_documents(user_query)

print("===检索结果===")
print(docs[0].page_content)

response = llm(
            template.format_messages(
                information=docs[0].page_content,
                query=user_query
            )
        )

print("===回答===")
print(response.content)

===检索结果===
is simple , high computational requirements have limited the development of LLMs to a few players. There have been public releases of pretrained LLMs (such as BLOOM (Scao et al., 2022), LLaMa-1
===回答===
There is no information available about a conversational variant of LLaMa-2.


### 2.3.1 向量检索初探

In [19]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS

embeddings = OpenAIEmbeddings() 
db = FAISS.from_documents(paragraphs, embeddings) #Facebook的开源向量检索引擎

user_query = "Does llama 2 have a conversational variant?"

docs = db.similarity_search(user_query)
print("===检索结果===")
print(docs[0].page_content)

response = llm(
            template.format_messages(
                information=docs[0].page_content,
                query=user_query
            )
        )

print("===回答===")
print(response.content)

===检索结果===
34 B variants , which were port on in this paper but are not releasing.§ 2.Llama 2-Chat , a fine-tuned version of Llama 2 that is optimized for dialogue use cases. We release variants of this model
===回答===
Yes, Llama 2 has a conversational variant called Llama 2-Chat. It is a fine-tuned version of Llama 2 that is optimized for dialogue use cases.


**你甚至可以跨语言检索**

In [20]:
user_query = "llama 2有对话式的版本吗"

docs = db.similarity_search(user_query)
print("===检索结果===")
print(docs[0].page_content)

response = llm(
            template.format_messages(
                information=docs[0].page_content,
                query=user_query
            )
        )

print("===回答===")
print(response.content)

===检索结果===
34 B variants , which were port on in this paper but are not releasing.§ 2.Llama 2-Chat , a fine-tuned version of Llama 2 that is optimized for dialogue use cases. We release variants of this model
===回答===
是的，Llama 2有一个专门针对对话场景进行优化的版本，称为Llama 2-Chat。


### 2.4 文档向量化：Text Embeddings

<div class="alert alert-success">
<b>Embedding：</b>将目标物体（词、句子、文章）表示成向量的方法
</div>


In [21]:
from langchain.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings() # 默认是text-embedding-ada-002
text = "这是一个测试"
document = "测试文档"
query_vec = embeddings.embed_query(text)
doc_vec = embeddings.embed_documents([document])

print(len(query_vec))
print(query_vec[:10])  # 为了展示方便，只打印前10维
print(len(doc_vec[0]))
print(doc_vec[0][:10])  # 为了展示方便，只打印前10维

1536
[-0.011436891812873518, -0.012987656599242918, 0.009020282942780396, -0.011973197646077209, -0.02477347121082455, 0.014486729257110757, -0.02189163318421738, -0.005188601123634472, -0.0012567658055167737, -0.0337032938600498]
1536
[-0.002583218747341917, 0.0008178391754881199, -0.0045899870032351945, -0.005638406631416479, -0.010556249915691514, 0.02594027304651135, -0.014526552570770228, -0.002125661361839841, -0.01431758986133824, -0.018316714156232938]


[**详细说说 Embedding**](embedding.ipynb)

### 2.5 向量的存储（与索引）：Vectorstores

<img src="vector_stores.jpg" style="margin-left: 0px" width=500px>


In [22]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
db = FAISS.from_documents(paragraphs, embeddings)

user_query = "where are the training data for llama 2 from"
docs = db.similarity_search(user_query)
print(docs[0].page_content)

#response = llm(template.format_messages(information=docs[0].page_content,query=user_query))
#print(f"===回答===\n{response.content}")


to the general public for research and commercial use‡: 1 . Llama 2 , an updated version of Llama 1 , trained on a new mix of publicly available data . We also increased the size of the pre training


**向量数据库功能对比**

<img src="vectordb.png" style="margin-left: 0px" width=500px>

### 2.6 向量检索：Retrievers


In [23]:
retriever = db.as_retriever()
docs = retriever.get_relevant_documents(user_query)

print(docs[0].page_content)

to the general public for research and commercial use‡: 1 . Llama 2 , an updated version of Llama 1 , trained on a new mix of publicly available data . We also increased the size of the pre training


### 2.6.1 Parent Document Retriever

从相关段落召回整个文档

In [24]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.storage import InMemoryStore
from langchain.docstore import InMemoryDocstore
from langchain.docstore.document import Document
import faiss

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=60,  
    length_function=len,
    add_start_index=True,
)

embedding_size = 1536 # OpenAIEmbeddings的维度
index = faiss.IndexFlatL2(embedding_size) # 精准检索
embedding_fn = OpenAIEmbeddings().embed_query
# 构造向量数据库
vectorstore = FAISS(embedding_fn, index, InMemoryDocstore({}), {})

# 文档存储
store = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore, 
    docstore=store, 
    child_splitter=text_splitter,
)

retriever.add_documents(pages[:4], ids=None)

user_query = "can llama2 be used for commercial purposes?"
sub_docs = vectorstore.similarity_search(user_query)
print("===段落===")
print(sub_docs[0].page_content)

retrieved_docs = retriever.get_relevant_documents(user_query)
print("===文档===")
print(retrieved_docs[0].page_content)


===段落===
2-Chatmodels.
We are releasing the following models to the general public for research and commercial use‡:
1.Llama 2 ,anupdatedversionof Llama 1,trainedonanewmixofpubliclyavailabledata. Wealso
===文档===
Figure 3: Safety human evaluation results for Llama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
importanttocaveatthesesafetyresultswiththeinherentbiasofLLMevaluationsduetolimitationsofthe
promptset,subjectivityofthereviewguidelines,andsubjectivityofindividualraters. Additionally,these
safety evaluations are performed using content standards that are likely to be biased towards the Llama
2-Chatmodels.
We are releasing the following models to the general public for research and commercial use‡:
1.Llama 2 ,anupdatedversionof Llama 1,trainedonanewmixofpubliclyavailabledata.

### 2.6.1 Reranker（选）

<img src="sbert-rerank.png" style="margin-left: 0px" width=500px>

Langchain里目前没有，但是实际生产中你会碰到需要 rerank 的情况!

支持Retrieval+Rerank的开源框架，参考：https://docs.jina.ai/

Langchain自带的机制:

- Long Context Recorder: https://python.langchain.com/docs/modules/data_connection/document_transformers/post_retrieval/long_context_reorder
- Ensemble Retriever (Hybrid Search): https://python.langchain.com/docs/modules/data_connection/retrievers/ensemble

## 三、记忆封装：Memory

### 3.1 对话上下文：ConversationBufferMemory


In [25]:
from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory

history = ConversationBufferMemory()
history.save_context({"input": "你好啊"}, {"output": "你也好啊"})

print(history.load_memory_variables({}))

history.save_context({"input": "你再好啊"}, {"output": "你又好啊"})

print(history.load_memory_variables({}))

{'history': 'Human: 你好啊\nAI: 你也好啊'}
{'history': 'Human: 你好啊\nAI: 你也好啊\nHuman: 你再好啊\nAI: 你又好啊'}


Message格式

In [26]:
from langchain.memory import ChatMessageHistory

history = ChatMessageHistory()

history.add_user_message("你好!")

history.add_ai_message("有什么可以帮您?")

print(history)

messages=[HumanMessage(content='你好!', additional_kwargs={}, example=False), AIMessage(content='有什么可以帮您?', additional_kwargs={}, example=False)]


只保留一个窗口的上下文


In [27]:
from langchain.memory import ConversationBufferWindowMemory

window = ConversationBufferWindowMemory(k=2)
window.save_context({"input": "第一轮问"}, {"output": "第一轮答"})
window.save_context({"input": "第二轮问"}, {"output": "第二轮答"})
window.save_context({"input": "第三轮问"}, {"output": "第三轮答"})
print(window.load_memory_variables({}))

{'history': 'Human: 第二轮问\nAI: 第二轮答\nHuman: 第三轮问\nAI: 第三轮答'}


### 3.2 自动对历史信息做摘要：ConversationSummaryMemory


In [3]:
from langchain.memory import ConversationSummaryMemory
from langchain.llms import OpenAI

memory = ConversationSummaryMemory(
    llm=OpenAI(temperature=0),
    # buffer="The conversation is between a customer and a sales."
    buffer="以中文表示"
)
memory.save_context(
    {"input": "你好"}, {"output": "你好，我是你的AI助手。我能为你回答有关AGIClass的各种问题。"})

print(memory.load_memory_variables({}))

{'history': '\n人类问AI助手你好，AI助手回答你好，表示自己是人类的AI助手，可以回答有关AGIClass的各种问题。'}


### 3.3 用向量数据库存储记忆

In [4]:
from datetime import datetime
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.llms import OpenAI
from langchain.memory import VectorStoreRetrieverMemory
from langchain.chains import ConversationChain
from langchain.prompts import PromptTemplate
import faiss

from langchain.docstore import InMemoryDocstore
from langchain.vectorstores import FAISS


embedding_size = 1536 # OpenAIEmbeddings的维度
index = faiss.IndexFlatL2(embedding_size)
embedding_fn = OpenAIEmbeddings().embed_query
vectorstore = FAISS(embedding_fn, index, InMemoryDocstore({}), {})

# 实际应用中k可以稍大一些，这里k=1演示方便
retriever = vectorstore.as_retriever(search_kwargs=dict(k=1))
memory = VectorStoreRetrieverMemory(retriever=retriever)

# 把记忆存在向量数据库中
memory.save_context({"input": "我喜欢看电影"}, {"output": "不错啊"})
memory.save_context({"input": "我喜欢踢足球"}, {"output": "下次一起啊"})
memory.save_context({"input": "我不喜欢喝咖啡"}, {"output": "ok"}) 

# 聊到相关话题，检索之前的记忆
print(memory.load_memory_variables({"prompt": "周末做点体育运动?"})["history"])

Retrying langchain.embeddings.openai.embed_with_retry.<locals>._embed_with_retry in 4.0 seconds as it raised RateLimitError: Rate limit reached for default-text-embedding-ada-002 in organization org-W6nlVyrVosglqBBTCNlsCLmW on requests per min. Limit: 3 / min. Please try again in 20s. Contact us through our help center at help.openai.com if you continue to have issues. Please add a payment method to your account to increase your rate limit. Visit https://platform.openai.com/account/billing to add a payment method..
Retrying langchain.embeddings.openai.embed_with_retry.<locals>._embed_with_retry in 4.0 seconds as it raised RateLimitError: Rate limit reached for default-text-embedding-ada-002 in organization org-W6nlVyrVosglqBBTCNlsCLmW on requests per min. Limit: 3 / min. Please try again in 20s. Contact us through our help center at help.openai.com if you continue to have issues. Please add a payment method to your account to increase your rate limit. Visit https://platform.openai.com/

input: 我喜欢踢足球
output: 下次一起啊


## 四、链架构：Chain


「Chains allow us to combine multiple components together to create a single, coherent application. For example, we can create a chain that takes user input, formats it with a PromptTemplate, and then passes the formatted response to an LLM. We can build more complex chains by combining multiple chains together, or by combining chains with other components.」

- Chain 封装了一个既定的流程
- 类比于函数封装了过程
- 建造者模式（Builder Pattern）, 解耦各种复杂的组件

### 4.1 一个最简单的 Chain


In [30]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.9)
prompt = PromptTemplate(
    input_variables=["product"],
    template="为生产{product}的公司取一个亮眼中文名字：",
)

chain = LLMChain(llm=llm, prompt=prompt)

print(chain.run("电脑"))

极致科技


### 4.2 在 Chain 中加入 Memory


In [5]:
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

template = """你是聊天机器人小瓜，你可以和人类聊天。

{memory}
Human: {human_input}
AI:"""

prompt = PromptTemplate(
    input_variables=["memory", "human_input"], template=template
)

# memory = ConversationBufferMemory(memory_key="memory")

memory = ConversationSummaryMemory(llm=OpenAI(
    temperature=0), buffer="以中文表示", memory_key="memory")

llm_chain = LLMChain(
    llm=OpenAI(),
    prompt=prompt,
    verbose=True,
    memory=memory,
)

print(llm_chain.run("你是谁？"))
print("---------------")
output = llm_chain.run("我刚才问了你什么，你是怎么回答的？")
print(output)



> Entering new LLMChain chain...
Prompt after formatting:
你是聊天机器人小瓜，你可以和人类聊天。

以中文表示
Human: 你是谁？
AI:

> Finished chain.
 我是聊天机器人小瓜，很高兴认识你！
---------------


> Entering new LLMChain chain...
Prompt after formatting:
你是聊天机器人小瓜，你可以和人类聊天。


以中文表示
人类问AI谁是，AI回答自己是聊天机器人小瓜，表示很高兴认识人类。
Human: 我刚才问了你什么，你是怎么回答的？
AI:

> Finished chain.


Retrying langchain.llms.openai.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised RateLimitError: Rate limit reached for default-text-davinci-003 in organization org-W6nlVyrVosglqBBTCNlsCLmW on requests per min. Limit: 3 / min. Please try again in 20s. Contact us through our help center at help.openai.com if you continue to have issues. Please add a payment method to your account to increase your rate limit. Visit https://platform.openai.com/account/billing to add a payment method..
Retrying langchain.llms.openai.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised RateLimitError: Rate limit reached for default-text-davinci-003 in organization org-W6nlVyrVosglqBBTCNlsCLmW on requests per min. Limit: 3 / min. Please try again in 20s. Contact us through our help center at help.openai.com if you continue to have issues. Please add a payment method to your account to increase your rate limit. Visit https://platform.openai.com/acco

 我回答说我是聊天机器人小瓜，很高兴认识你。


### 4.3 一个复杂一点的 Chain


<img src="https://raw.githubusercontent.com/showkawa/raise-issue-use-picture/master/blog/cnblogs/chain/stuffdocchain.png" style="margin-left: 0px;background-color:#fdfdfe" width=800px>


In [ ]:
!pip install unstructured faiss-cpu


In [32]:
from langchain.document_loaders import UnstructuredMarkdownLoader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.llms import OpenAI
from langchain.chains import RetrievalQA
from langchain.document_loaders import PyPDFLoader

loader = PyPDFLoader("llama2.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=60,
    length_function=len,
    add_start_index=True,
)

texts = text_splitter.create_documents([pages[2].page_content,pages[3].page_content])

embeddings = OpenAIEmbeddings()
db = FAISS.from_documents(texts, embeddings)

qa_chain = RetrievalQA.from_chain_type(
    llm=OpenAI(temperature=0), #语言模型
    chain_type="stuff",  # prompt的组织方式，后面细讲
    retriever=db.as_retriever() # 检索器
)

query = "llama 2能商用吗"
response = qa_chain.run(query)
print(response)

 Yes, Llama 2-Chat models are available for research and commercial use.


In [33]:
print('================qa_chain===============')
print(qa_chain)
print('======combine_documents_chain==========')
print(qa_chain.combine_documents_chain.document_prompt)
print('==============llm_chain================')
print(qa_chain.combine_documents_chain.llm_chain.prompt.template)

================qa_chain===============
memory=None callbacks=None callback_manager=None verbose=False tags=None metadata=None combine_documents_chain=StuffDocumentsChain(memory=None, callbacks=None, callback_manager=None, verbose=False, tags=None, metadata=None, input_key='input_documents', output_key='output_text', llm_chain=LLMChain(memory=None, callbacks=None, callback_manager=None, verbose=False, tags=None, metadata=None, prompt=PromptTemplate(input_variables=['context', 'question'], output_parser=None, partial_variables={}, template="Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:", template_format='f-string', validate_template=True), llm=OpenAI(cache=None, verbose=False, callbacks=None, callback_manager=None, tags=None, metadata=None, client=<class 'openai.api_resources.completion.Completion'>, model_name='text-d

### 4.4 常用的基础 Chain 类型：Sequential


In [35]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SimpleSequentialChain

llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
name_prompt = PromptTemplate(
    input_variables=["query"],
    template="从给定句子中提取出完整地址：{query}\n直接输出结果。",
)

name_chain = LLMChain(llm=llm, prompt=name_prompt)

slogan_prompt = PromptTemplate(
    input_variables=["address"],
    template="将'{address}'翻译成英文\n直接输出结果。",
)

slogan_chain = LLMChain(llm=llm, prompt=slogan_prompt)

overall_chain = SimpleSequentialChain(
    chains=[name_chain, slogan_chain], verbose=True)

print(overall_chain.run("收件地址北京市朝阳区东方东路19号"))



> Entering new SimpleSequentialChain chain...
北京市朝阳区东方东路19号
19 Dongfang East Road, Chaoyang District, Beijing City.

> Finished chain.
19 Dongfang East Road, Chaoyang District, Beijing City.


### 4.5 常用的基础 Chain 类型：Transform


In [36]:
import re
from langchain.chains import TransformChain, LLMChain, SimpleSequentialChain
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

# 例如：发给OpenAI之前，把用户隐私数据抹掉
def anonymize(inputs: dict) -> dict:
    text = inputs["text"]
    t = re.compile(
        r'1(3\d|4[4-9]|5[0-35-9]|6[67]|7[013-8]|8[0-9]|9[0-9])\d{8}')
    while True:
        s = re.search(t, text)
        if s:
            text = text.replace(s.group(), '***********')
        else:
            break
    return {"output_text": text}


transform_chain = TransformChain(
    input_variables=["text"], output_variables=["output_text"], transform=anonymize
)

llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.9)
prompt = PromptTemplate(
    input_variables=["input"],
    template="根据下述句子，提取候选人的职业:\n{input}\n输出JSON, 以job为key",
)

task_chain = LLMChain(llm=llm, prompt=prompt)

overall_chain = SimpleSequentialChain(
    chains=[transform_chain, task_chain], verbose=True)

print(overall_chain.run("我是程序员，有事随时跟我联系，打我手机13912345678"))



> Entering new SimpleSequentialChain chain...
我是程序员，有事随时跟我联系，打我手机***********
{"job": "程序员"}

> Finished chain.
{"job": "程序员"}


### 4.6 常用的基础 Chain 类型：Router


In [37]:
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.prompts import PromptTemplate
from langchain.chains.llm import LLMChain
from langchain.chains import ConversationChain
from langchain.llms import OpenAI
from langchain.chains.router import MultiPromptChain
import warnings
warnings.filterwarnings("ignore")


windows_template = """
你只会写DOS或Windows Shell脚本。你不会写任何其他语言的程序。你也不会写Linux脚本。

用户问题:
{input}
"""

linux_template = """
你只会写Linux Shell脚本。你不会写任何其他语言的程序。你也不会写Windows脚本。

用户问题:
{input}
"""

prompt_infos = [
    {
        "name": "WindowsExpert",
        "description": "擅长回答Windows Shell相关问题",
        "prompt_template": windows_template,
    },
    {
        "name": "LinuxExpert",
        "description": "擅长回答Linux Shell相关问题",
        "prompt_template": linux_template,
    },
]

llm = OpenAI()

destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=["input"]
    )
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain
    
default_chain = ConversationChain(llm=llm, output_key="text")

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=default_chain,
    verbose=True,
)

print(chain.run("帮我写个脚本，让Windows系统每天0点自动校对时间"))

print(chain.run("帮我写个cron脚本，让系统每天0点自动重启"))



> Entering new MultiPromptChain chain...
WindowsExpert: {'input': '请帮我写一个脚本，让Windows系统每天0点自动校对时间'}
> Finished chain.

答案:
使用Windows自带的schtasks.exe命令，可以在每天0点自动运行一个命令来校对时间：

schtasks /create /tn "Time Check" /tr "w32tm /resync" /sc daily /st 00:00

上面的命令会创建一个名为“Time Check”的定时任务，每天0点运行w32tm /resync命令来校对时间。


> Entering new MultiPromptChain chain...
LinuxExpert: {'input': '帮我写个cron脚本，让系统每天0点自动重启'}
> Finished chain.

答案:
#!/bin/bash
#This script will restart the system everyday at 0:00

crontab -e
0 0 * * * /sbin/shutdown -r now


<div class="alert alert-warning">
<b>思考：</b>Router是否是一个必要的基础类型？
</div>


### 4.7 调用 OpenAI Function Calling 获得 Pydantic 输出


In [42]:
from pydantic import BaseModel, Field
from typing import Optional
from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.schema import HumanMessage, SystemMessage

from langchain.chains.openai_functions import (
    create_openai_fn_chain,
    create_structured_output_chain,
)
from langchain.chat_models import ChatOpenAI
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate


class Contact(BaseModel):
    """抽取联系人信息"""

    name: str = Field(..., description="联系人姓名")
    address: str = Field(..., description="联系人地址")
    tel: str = Field(None, description="联系人电话")


prompt_msgs = [
    SystemMessage(
        content="你是信息抄录员。"
    ),
    HumanMessage(
        content="根据给定个数从下面的句子中抽取信息:"
    ),
    HumanMessagePromptTemplate.from_template("{input}"),
    HumanMessage(content="Tips: Make sure to answer in the correct format"),
]
prompt = ChatPromptTemplate(messages=prompt_msgs)
llm = ChatOpenAI(model="gpt-4-0613", temperature=0)

chain = create_openai_fn_chain([Contact], llm, prompt, verbose=True)

chain.run("寄给亮马桥外交办公大楼的王卓然，13012345678")



> Entering new LLMChain chain...
Prompt after formatting:
System: 你是信息抄录员。
Human: 根据给定个数从下面的句子中抽取信息:
Human: 寄给亮马桥外交办公大楼的王卓然，13012345678
Human: Tips: Make sure to answer in the correct format

> Finished chain.


Contact(name='王卓然', address='亮马桥外交办公大楼', tel='13012345678')

### 4.8 基于 Document 的 Chains

<img src="stuff.jpg" style="margin-left: 0px" width=500px>
<img src="refine.jpg" style="margin-left: 0px" width=500px>
<img src="map_reduce.jpg" style="margin-left: 0px" width=500px>
<img src="map_rerank.jpg" style="margin-left: 0px" width=500px>


In [3]:
from langchain.callbacks import StdOutCallbackHandler
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.chains.base import Chain
from langchain.document_loaders import PyPDFLoader

# 把整个chain的verbose打开
def set_verbose_recusively(chain):
    chain.verbose = True
    for attr in dir(chain):
        if attr.endswith('_chain') and isinstance(getattr(chain, attr), Chain):
            subchain = getattr(chain, attr)
            set_verbose_recusively(subchain)


loader = PyPDFLoader("llama2.pdf")
documents = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    length_function=len,
    add_start_index=True,
)

paragraphs = text_splitter.create_documents(
    [preprocess(d.page_content) for d in documents[2:4]])
# print(paragraphs)
embeddings = OpenAIEmbeddings(model='text-embedding-ada-002')
db = FAISS.from_documents(paragraphs, embeddings)
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(temperature=0),
    chain_type="stuff",
    retriever=db.as_retriever(),
    verbose=True
)
set_verbose_recusively(qa_chain)

query = "how many parameters does the smallest version of llama 2 contain?"
qa_chain.run(query)



> Entering new RetrievalQA chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the users question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
that are likely to be biased towards the Llama 2-Chatmodels. We are releasing the following models to the general public for research and commercial use‡: 1 . Llama 2 , an updated version of Llama 1

fine-tuned LLMs, Llama 2 and Llama 2-Chat , at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested, Llama 2-Chat models generally perform better than existing

by 40 % , doubled the context length of the model , and adopted grouped query attention ( A in s lie et al . , 2023 ) . We are releasing variants of Llama 2 with 7 B , 13 B , and 70 B parameters . We

testing and tuning tailored to their specific applications of the model .

'The smallest version of Llama 2 contains 7 billion parameters.'

## 五、智能体架构：Agent


### 5.1 什么是智能体（Agent）

将大语言模型作为一个推理引擎。给定一个任务，智能体自动生成完成任务所需的步骤，执行相应动作（例如选择并调用工具），直到任务完成。


### 5.2 先定义一些工具：Tools

- 可以是一个函数或三方 API
- 也可以把一个 Chain 或者 Agent 的 run()作为一个 Tool


In [4]:
from langchain import SerpAPIWrapper
from langchain.tools import Tool, tool

search = SerpAPIWrapper()
tools = [
    Tool.from_function(
        func=search.run,
        name="Search",
        description="useful for when you need to answer questions about current events"
    ),
]

In [5]:
import calendar
import dateutil.parser as parser
from datetime import date


@tool("weekday")
def weekday(date_str: str) -> str:
    """Convert date to weekday name"""
    d = parser.parse(date_str)
    return calendar.day_name[d.weekday()]

In [6]:
from langchain.agents import load_tools

tools = load_tools(["serpapi"])
tools += [weekday]

### 5.3 智能体类型：ReAct


<img src="ReAct.png" style="margin-left: 0px" width=500px>


In [ ]:
!pip install google-search-results


In [7]:
from langchain.chat_models import ChatOpenAI
from langchain.llms import OpenAI
from langchain.agents import AgentType
from langchain.agents import initialize_agent

llm = ChatOpenAI(model_name='gpt-4', temperature=0)

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)
agent.run("周杰伦生日那天是星期几")



> Entering new AgentExecutor chain...
I need to find out when Jay Chou's birthday is and then use the weekday function to determine what day of the week it falls on.
Action: Search
Action Input: "Jay Chou birthday"
Observation: January 18, 1979
Thought:Now that I know Jay Chou's birthday, I can use the weekday function to find out what day of the week it falls on.
Action: weekday
Action Input: "1979-01-18"
Observation: Thursday
Thought:I now know the final answer
Final Answer: 周杰伦的生日是在星期四。

> Finished chain.


'周杰伦的生日是在星期四。'

### 5.4 通过 OpenAI Function Calling 实现智能体


In [50]:
from langchain.chat_models import ChatOpenAI
from langchain.llms import OpenAI
from langchain.agents import AgentType
from langchain.agents import initialize_agent

llm = ChatOpenAI(model_name='gpt-4-0613', temperature=0)

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True,
    max_iterations=2,
    early_stopping_method="generate",
)
agent.run("周杰伦生日那天是星期几")



> Entering new AgentExecutor chain...

Invoking: `Search` with `周杰伦的生日`


January 18, 1979
Invoking: `weekday` with `{'date_str': '1979-01-18'}`


Thursday周杰伦的生日（1979年1月18日）是星期四。

> Finished chain.


'周杰伦的生日（1979年1月18日）是星期四。'

### 5.5 智能体类型：SelfAskWithSearch


In [8]:
from langchain import OpenAI, SerpAPIWrapper
from langchain.agents import initialize_agent, Tool
from langchain.agents import AgentType

llm = OpenAI(temperature=0)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Intermediate Answer",
        func=search.run,
        description="useful for when you need to ask with search",
    )
]

self_ask_with_search = initialize_agent(
    tools, llm, agent=AgentType.SELF_ASK_WITH_SEARCH, verbose=True
)
self_ask_with_search.run(
    "吴京的老婆的主持过什么节目"
)



> Entering new AgentExecutor chain...
 Yes.
Follow up: 吴京的老婆是谁？
Intermediate answer: Xie Nan
Follow up: Xie Nan主持过什么节目？
Intermediate answer: 现为光线传媒旗下主打节目《娱乐现场》、《最佳现场》、《影视风云榜》当家主持，更是黑龙江卫视《快活武林》、深圳卫视《大牌生日会》、辽宁卫视《谁是主角》等节目女主持。
So the final answer is: 光线传媒旗下主打节目《娱乐现场》、《最佳现场》、《影视风云榜》当家主持，更是黑龙江卫视《快活武林》、深圳卫视《大牌生日会》、辽宁卫视《谁是主角》等节目女主持。

> Finished chain.


'光线传媒旗下主打节目《娱乐现场》、《最佳现场》、《影视风云榜》当家主持，更是黑龙江卫视《快活武林》、深圳卫视《大牌生日会》、辽宁卫视《谁是主角》等节目女主持。'

### 5.6 智能体类型：Plan-and-Execute


<img src="PlanExec.png" style="margin-left: 0px" width=500px>


In [ ]:
!pip install langchain-experimental


In [10]:
from langchain.utilities.wolfram_alpha import WolframAlphaAPIWrapper
from langchain.agents import load_tools
from langchain import SerpAPIWrapper
from langchain.agents.tools import Tool
from langchain.tools import StructuredTool
from langchain.llms import OpenAI
from langchain_experimental.plan_and_execute import PlanAndExecute, load_agent_executor, load_chat_planner
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryMemory
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from pydantic import BaseModel, BaseSettings, Field

class TranslateArguments(BaseModel):
    text: str = Field(default="", description="source text to translate")
    language: str = Field(default="", description="target language to translate into")

llm = ChatOpenAI(model_name='gpt-4', temperature=0)

search = SerpAPIWrapper()

prompt = PromptTemplate(
    input_variables=["text","language"],
    template="{text}\n\nTranslate the above text into {language}. Generate output directly without comments or acknowledgements.",
)

translation_chain = LLMChain(llm=llm, prompt=prompt)

tools = [
    StructuredTool(
        name="Translate",
        func=translation_chain.run,
        description="useful for when you need to translate",
        args_schema=TranslateArguments,
    ),
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events"
    )
]

planner = load_chat_planner(llm)
executor = load_agent_executor(llm, tools, verbose=True)
agent = PlanAndExecute(planner=planner, executor=executor, verbose=True)

agent.run("写一份关于开源LLM(large language model)的中文简报")



> Entering new PlanAndExecute chain...
steps=[Step(value='Research about the open-source Large Language Model (LLM). Understand its features, uses, benefits, and limitations.'), Step(value='Translate the gathered information into Chinese.'), Step(value='Structure the information in a brief format. Start with an introduction about LLM, followed by its features, uses, benefits, and limitations.'), Step(value='Write the brief in a clear and concise manner, ensuring the information is easily understandable.'), Step(value='Review the brief for any errors or inconsistencies.'), Step(value='Given the above steps taken, write the brief about the open-source Large Language Model in Chinese.\n')]

> Entering new AgentExecutor chain...
Thought: The user wants to know about the open-source Large Language Model (LLM). I will use the Search tool to find information about its features, uses, benefits, and limitations.

Action:
```
{
  "action": "Search",
  "action_input": "open-source Large Languag

'大型语言模型（LLMs）是强大的AI工具，具有多种应用。它们以速度、适应性、隐私和能力而闻名，但也需要大量资源进行训练和推理。LLMs可以用于翻译、回答问题和生成文本等任务。它们可以针对各种任务进行微调，使其成为AI项目的多功能工具。开源的LLMs提供了增强安全性和隐私的好处，因为它们允许组织内部保留敏感数据。然而，它们也有局限性。它们可能过度依赖训练数据，这可能导致在训练分布之外的数据上表现不佳。此外，它们的工作方式可能难以解释，这在理解它们如何生成响应时带来了挑战。'

## 六、Callbacks

回调函数，用于监测、记录调用过程中的信息


In [ ]:
class BaseCallbackHandler:
    """Base callback handler that can be used to handle callbacks from langchain."""

    def on_llm_start(
        self, serialized: Dict[str, Any], prompts: List[str], **kwargs: Any
    ) -> Any:
        """Run when LLM starts running."""

    def on_chat_model_start(
        self, serialized: Dict[str, Any], messages: List[List[BaseMessage]], **kwargs: Any
    ) -> Any:
        """Run when Chat Model starts running."""

    def on_llm_new_token(self, token: str, **kwargs: Any) -> Any:
        """Run on new LLM token. Only available when streaming is enabled."""

    def on_llm_end(self, response: LLMResult, **kwargs: Any) -> Any:
        """Run when LLM ends running."""

    def on_llm_error(
        self, error: Union[Exception, KeyboardInterrupt], **kwargs: Any
    ) -> Any:
        """Run when LLM errors."""

    def on_chain_start(
        self, serialized: Dict[str, Any], inputs: Dict[str, Any], **kwargs: Any
    ) -> Any:
        """Run when chain starts running."""

    def on_chain_end(self, outputs: Dict[str, Any], **kwargs: Any) -> Any:
        """Run when chain ends running."""

    def on_chain_error(
        self, error: Union[Exception, KeyboardInterrupt], **kwargs: Any
    ) -> Any:
        """Run when chain errors."""

    def on_tool_start(
        self, serialized: Dict[str, Any], input_str: str, **kwargs: Any
    ) -> Any:
        """Run when tool starts running."""

    def on_tool_end(self, output: str, **kwargs: Any) -> Any:
        """Run when tool ends running."""

    def on_tool_error(
        self, error: Union[Exception, KeyboardInterrupt], **kwargs: Any
    ) -> Any:
        """Run when tool errors."""

    def on_text(self, text: str, **kwargs: Any) -> Any:
        """Run on arbitrary text."""

    def on_agent_action(self, action: AgentAction, **kwargs: Any) -> Any:
        """Run on agent action."""

    def on_agent_finish(self, finish: AgentFinish, **kwargs: Any) -> Any:
        """Run on agent end."""

In [59]:
from langchain.callbacks import StdOutCallbackHandler
from langchain.callbacks.base import BaseCallbackHandler
from langchain.chains import LLMChain
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from typing import List, Dict, Any
from langchain.schema.output import LLMResult

class myhandler(BaseCallbackHandler):
    def on_llm_start(
        self, serialized: Dict[str, Any], prompts: List[str], **kwargs: Any
    ) -> Any:
        print(f"\x1b[34mFeed LLM with {prompts}\x1b[0m")

    def on_llm_end(self, response: LLMResult, **kwargs: Any) -> Any:
        print(f"\x1b[34mLLM Returns: {response}\x1b[0m")

    def on_chain_start(
        self, serialized: Dict[str, Any], inputs: Dict[str, Any], **kwargs: Any
    ) -> Any:
        print(f"Chain Starts: {inputs}")

    def on_chain_end(self, outputs: Dict[str, Any], **kwargs: Any) -> Any:
        print(f"Chain Ends!")

    def on_text(self, text: str, **kwargs: Any) -> Any:
        print(f"On text: {text}")


handler = myhandler()

llm = OpenAI()
prompt = PromptTemplate.from_template("1 + {number} = ")

# 在构造的时候加回调，只触发这个对象对应的事件
chain = LLMChain(llm=llm, prompt=prompt, callbacks=[handler])
x = chain.run(number=1)
print("----------")
# 在运行的时候加回调，触发这个过程的所有子过程的事件
chain = LLMChain(llm=llm, prompt=prompt)
x = chain.run(number=2, callbacks=[handler])

Chain Starts: {'number': 1}
On text: Prompt after formatting:
1 + 1 = 
Chain Ends!
----------
Chain Starts: {'number': 2}
On text: Prompt after formatting:
1 + 2 = 
Feed LLM with ['1 + 2 = ']
LLM Returns: generations=[[Generation(text='\n\n3', generation_info={'finish_reason': 'stop', 'logprobs': None})]] llm_output={'token_usage': {'prompt_tokens': 5, 'completion_tokens': 3, 'total_tokens': 8}, 'model_name': 'text-davinci-003'} run=None
Chain Ends!


## 七、大模型时代软件的演变趋势

### 7.1 从形式逻辑向思维逻辑过渡

<img src="agent.png" style="margin-left: 0px" width=600px>

### 7.2 从单智能体到多智能体协作

<img src="metagpt.png" style="margin-left: 0px" width=600px>

<div class="alert alert-warning">
<b>思考：</b>
<ul>
<li>从软件工程的角度，Langchain现阶段的缺点是什么</li>
<li>距离智能体大规模应用，我们还有什么没解决的问题</li>
<li>你觉得智能体有哪些可优化的方向</li>
</ul>
</div>


## LangFlow

<img src="https://github.com/logspace-ai/langflow/raw/main/img/langflow-demo.gif?raw=true" style="margin-left: 0px">

https://github.com/logspace-ai/langflow


## 作业

做个自己的 [ChatPDF(https://www.chatpdf.com/)](https://www.chatpdf.com/)（需科学访问）。需求：

1. 从本地加载 PDF 文件，基于 PDF 的内容对话
2. 可以无前端，只要能在命令行运行就行
3. 其它随意发挥
